## Smoothing (E-W neighbors avg)

In [1]:
%%writefile q5.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 60
#define LOW 10
#define HIGH 99
#define BLOCK_SIZE 8

// ── CUDA Kernel
__global__ void smoothKernel(int *dA, int *dC, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        int left  = (i > 0)     ? dA[i - 1] : dA[i];
        int right = (i < n - 1) ? dA[i + 1] : dA[i];
        dC[i] = (left + dA[i] + right) / 3;
    }
}

// ── Host Function
__host__ void smooth(int *hA, int *hC, int n)
{
    int *dA, *dC;
    int size = n * sizeof(int);
    cudaMalloc((void**)&dA, size);
    cudaMalloc((void**)&dC, size);
    cudaMemcpy(dA, hA, size, cudaMemcpyHostToDevice);
    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);
    smoothKernel<<<DimGrid, DimBlock>>>(dA, dC, n);
    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);
    cudaFree(dA);
    cudaFree(dC);
}

// ── Main
int main()
{
    int *hA, *hC;
    hA = (int*) malloc(N * sizeof(int));
    hC = (int*) malloc(N * sizeof(int));

    srand(time(NULL));
    for (int i = 0; i < N; i++)
        hA[i] = rand() % (HIGH - LOW + 1) + LOW;

    printf("hA:\n");
    for (int i = 0; i < N; i++) printf("%4d", hA[i]);
    printf("\n");

    smooth(hA, hC, N);

    printf("\nhC (Smoothed):\n");
    for (int i = 0; i < N; i++) printf("%4d", hC[i]);
    printf("\n");

    free(hA); free(hC);
    return 0;
}

Writing q5.cu


In [2]:
!nvcc -arch=sm_75 q5.cu -o q5

!./q5

hA:
  18  41  66  76  79  53  80  60  13  45  10  14  27  52  44  60  39  17  85  11  69  27  70  48  85  21  20  11  56  81  53  64  22  20  41  91  25  21  13  28  18  14  85  87  18  81  48  47  89  85  11  20  12  71  58  87  44  30  89  90

hC (Smoothed):
  25  41  61  73  69  70  64  51  39  22  23  17  31  41  52  47  38  47  37  55  35  55  48  67  51  42  17  29  49  63  66  46  35  27  50  52  45  19  20  19  20  39  62  63  62  49  58  61  73  61  38  14  34  47  72  63  53  54  69  89
